# Laboratorio 5 · Limpieza y preprocesamiento de los tweets

**Curso:** CC3084 · Data Science
**Entrada:** `data/raw/train.csv` y `data/raw/test.csv`
**Salida:** `data/processed/*_limpio_lema_2gramas.csv`

Este notebook aplica el pipeline de limpieza definido en `src/limpieza/` a los
datos crudos y guarda el resultado en `data/processed`. El nombre del archivo de
salida deja explícito **cuántos n-gramas** se usaron, para poder generar y
comparar versiones (`..._2gramas.csv`, `..._3gramas.csv`) sin sobrescribirlas.

**Decisiones tomadas a partir del EDA** (ver `analisis_exploratorio.ipynb`):

- **No se balancean las clases.** El EDA mostró 57% / 43%, un desbalance leve
  que no justifica remuestreo; basta con reportar F1-score al evaluar.
- **Se conservan hashtags y menciones** (sin el símbolo `#`/`@`) y además se
  les adjunta un flag que recuerda su origen, porque la palabra del hashtag
  suele ser justamente el término del desastre.
- **Se guarda un flag de link.** El EDA mostró que 66% de los tweets de desastre
  real traen URL vs 41% de los que no: es señal aprovechable.
- **Los números se resumen en flags**, separando el `911` (llamada de
  emergencia) del resto, en vez de borrarlos o dejar miles de cifras sueltas
  en el vocabulario.
- **Se usan bigramas.** Palabras como `fire`, `body` o `emergency` aparecen con
  frecuencia alta en *ambas* clases; el contexto de dos palabras
  (`forest_fire`, `body_bag`) es lo que desambigua.


## 1. Configuración

`N_NGRAMAS = 2` es el parámetro central del notebook: se propaga tanto al
procesamiento como al nombre del archivo de salida. Cambiarlo aquí y volver a
ejecutar genera una versión nueva sin tocar la anterior.


In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..").resolve() / "src"))

from limpieza.eliminacion_palabras_vacias import (
    NEGACIONES,
    construir_corpus_stopwords,
    eliminar_palabras_vacias,
)
from limpieza.lem import lematizar
from limpieza.limpieza_ruido import (
    FLAG_HASHTAG,
    FLAG_MENCION,
    limpiar_ruido_previo,
    quitar_puntuacion,
)
from limpieza.n_gramas import generar_hasta_n, generar_ngramas
from limpieza.normalizacion import (
    FLAG_EMERGENCIA,
    FLAG_NUMERO,
    marcar_numeros,
    normalizar,
)
from limpieza.pipeline import guardar_procesado, procesar_dataframe, procesar_texto
from limpieza.tokenizar import tokenizar

pd.set_option("display.max_colwidth", 120)

N_NGRAMAS = 2          # 2 = unigramas + bigramas
METODO_RAIZ = "lema"   # 'lema' (WordNet) o 'stem' (Porter)

DIR_RAW = Path("..") / "data" / "raw"
DIR_PROCESSED = Path("..") / "data" / "processed"

print(f"n-gramas: hasta {N_NGRAMAS} | reducción: {METODO_RAIZ}")


n-gramas: hasta 2 | reducción: lema


In [2]:
train = pd.read_csv(DIR_RAW / "train.csv")
test = pd.read_csv(DIR_RAW / "test.csv")

print(f"train: {train.shape}")
print(f"test:  {test.shape}")
train.head(3)


train: (7613, 5)
test:  (3263, 4)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place or...,1


## 2. Los pasos del pipeline, uno por uno

Antes de correr todo junto se muestra qué hace cada módulo sobre un tweet real,
para que el proceso sea auditable y no una caja negra.

El **orden importa**: la normalización va primero porque expandir `don't` →
`do not` necesita el apóstrofe, que el paso de limpieza de ruido elimina junto
con el resto de la puntuación.


In [3]:
ejemplo = train.loc[train["text"].str.contains("http", na=False), "text"].iloc[3]
print("ORIGINAL:")
print(ejemplo)


ORIGINAL:
On plus side LOOK AT THE SKY LAST NIGHT IT WAS ABLAZE http://t.co/qqsmshaJ3N


### Paso 1 · Normalización (`normalizacion.py`)

Minúsculas, corrección de artefactos de codificación (`\x89Û_`), expansión de
contracciones y reducción de alargamientos (`soooo` → `soo`).


In [4]:
paso1 = normalizar(ejemplo)
print(paso1)


on plus side look at the sky last night it was ablaze http://t.co/qqsmshaj3n


### Paso 2 · Limpieza de ruido (`limpieza_ruido.py`)

Quita el link (dejando un espacio para no pegar la palabra siguiente) y levanta
el flag, y elimina emojis.

A menciones y hashtags se les quita el símbolo pero **se les adjunta un flag
delante**, de modo que no se pierda de dónde venía la palabra:

| En el tweet | Queda como | Bigrama resultante |
|---|---|---|
| `#earthquake` | `flaghashtag earthquake` | `flaghashtag_earthquake` |
| `@bbcmtd` | `flagmencion bbcmtd` | `flagmencion_bbcmtd` |

El flag va **delante** y como token aparte (en vez de pegarlo formando
`hashtagearthquake`) por dos razones: el bigrama queda legible y captura la
relación completa, y el unigrama `earthquake` **sigue existiendo por separado**,
sumando a la misma feature que cuando esa palabra aparece sin `#`. Si se
fusionaran en un solo token, el modelo trataría `#earthquake` y `earthquake`
como palabras sin relación entre sí.

La puntuación **todavía no** se elimina aquí: el paso 3 necesita leer números
como `13,000` completos, y quitarles la coma antes los partiría en dos
(`13` y `000`), generando flags duplicados.


In [5]:
paso2, flags_ruido = limpiar_ruido_previo(paso1)
print(paso2)
print(f"\n{flags_ruido}")


on plus side look at the sky last night it was ablaze

{'tiene_link': True, 'tiene_mencion': False, 'tiene_hashtag': False}


In [6]:
# El flag conserva el origen de la palabra sin sacrificar el unigrama
for demo_marca in ["Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all",
                   "@bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C"]:
    texto_marcado, _ = limpiar_ruido_previo(normalizar(demo_marca))
    print(f"ANTES  : {demo_marca}")
    print(f"DESPUÉS: {texto_marcado}\n")


ANTES  : Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
DESPUÉS: our deeds are the reason of this flaghashtag earthquake may allah forgive us all

ANTES  : @bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C
DESPUÉS: flagmencion bbcmtd wholesale markets ablaze



### Paso 3 · Números y flag de emergencia (`normalizacion.py`)

En vez de borrar los números se sustituyen por flags, distinguiendo el **911**
(número de emergencias en EE.UU., de donde proviene la mayoría de los tweets)
del resto de cifras:

| Número en el tweet | Se reemplaza por |
|---|---|
| `911` | `flagnumero flagemergencia` |
| cualquier otro | `flagnumero flagnoemergencia` |

Así el modelo recibe la señal *"aquí había un número y era una llamada de
emergencia"* sin tener que aprender un vocabulario de miles de cifras
distintas, que solo aportarían ruido.

Este paso va **después** de quitar los links: si no, los dígitos que vienen
dentro de una URL (`t.co/lHYXEOHY6C`) se marcarían como números del tweet.


In [7]:
paso3, flag_911 = marcar_numeros(paso2)
paso3 = quitar_puntuacion(paso3)
print(paso3)
print(f"\n¿mencionaba el 911? -> {flag_911}")


on plus side look at the sky last night it was ablaze

¿mencionaba el 911? -> False


In [8]:
# Demostración del criterio de números sobre casos construidos
for demo_num in ["Call 911 now! Building on fire",
                 "13,000 people receive evacuation orders",
                 "@user check http://t.co/aB3x9Y link"]:
    texto_flags, es_911 = marcar_numeros(limpiar_ruido_previo(normalizar(demo_num))[0])
    print(f"ANTES : {demo_num}")
    print(f"FLAGS : {quitar_puntuacion(texto_flags)}")
    print(f"911   : {es_911}\n")


ANTES : Call 911 now! Building on fire
FLAGS : call flagnumero flagemergencia now  building on fire
911   : True

ANTES : 13,000 people receive evacuation orders
FLAGS : flagnumero flagnoemergencia people receive evacuation orders
911   : False

ANTES : @user check http://t.co/aB3x9Y link
FLAGS : flagmencion user check link
911   : False



### Paso 4 · Tokenización (`tokenizar.py`)

In [9]:
paso4_tokens = tokenizar(paso3)
print(paso4_tokens)


['on', 'plus', 'side', 'look', 'at', 'the', 'sky', 'last', 'night', 'it', 'was', 'ablaze']


### Paso 5 · Eliminación de palabras vacías (`eliminacion_palabras_vacias.py`)

Se parte del corpus de stopwords en inglés de NLTK, se le suma ruido de Twitter
(`rt`, `amp`, `lol`…) y se le **restan las negaciones**: NLTK considera stopword
a `no` y `not`, pero en este problema invierten el sentido de la frase
(`no fire` ≠ `fire`).


In [10]:
corpus_stopwords = construir_corpus_stopwords()

print(f"Stopwords en el corpus final: {len(corpus_stopwords)}")
print(f"Negaciones preservadas: {len(NEGACIONES)}")
print(f"\n¿'not' se elimina? -> {'not' in corpus_stopwords}")
print(f"¿'the' se elimina?  -> {'the' in corpus_stopwords}")

paso5 = eliminar_palabras_vacias(paso4_tokens, corpus_stopwords)
print(f"\n{paso5}")


Stopwords en el corpus final: 185
Negaciones preservadas: 32

¿'not' se elimina? -> False
¿'the' se elimina?  -> True

['plus', 'side', 'look', 'sky', 'last', 'night', 'ablaze']


In [11]:
# Comprobación explícita de que la negación sobrevive al pipeline
demo = "There's no fire here, don't panic!!!"
r_demo = procesar_texto(demo, corpus_stopwords, n_ngramas=N_NGRAMAS)
print("ANTES  :", demo)
print("DESPUÉS:", r_demo["texto_limpio"])


ANTES  : There's no fire here, don't panic!!!
DESPUÉS: no fire not panic


### Paso 6 · Lematización (`lem.py`)

Reduce cada palabra a su forma de diccionario usando su categoría gramatical
(`fires` → `fire`, `burning` → `burn`). Se prefiere sobre el stemming de Porter
(`stem.py`) porque devuelve palabras reales, lo que mantiene legibles los
n-gramas y la nube de palabras.


In [12]:
paso6 = lematizar(paso5)
print(paso6)


['plus', 'side', 'look', 'sky', 'last', 'night', 'ablaze']


### Paso 7 · N-gramas (`n_gramas.py`)

Con `N_NGRAMAS = 2` se combinan unigramas + bigramas. Los tokens de cada
bigrama se unen con `_` para que funcione como un único término al vectorizar.


In [13]:
solo_bigramas = generar_ngramas(paso6, 2)
paso7 = generar_hasta_n(paso6, N_NGRAMAS)

print(f"Unigramas ({len(paso6)}): {paso6}")
print(f"\nBigramas ({len(solo_bigramas)}): {solo_bigramas}")
print(f"\nTotal de términos con n<={N_NGRAMAS}: {len(paso7)}")


Unigramas (7): ['plus', 'side', 'look', 'sky', 'last', 'night', 'ablaze']

Bigramas (6): ['plus_side', 'side_look', 'look_sky', 'sky_last', 'last_night', 'night_ablaze']

Total de términos con n<=2: 13


## 3. Ejecución del pipeline sobre todo el dataset

`procesar_dataframe` encadena los siete pasos y agrega cada etapa intermedia
como columna, de modo que el resultado sea auditable fila por fila.


In [14]:
%%time
train_limpio = procesar_dataframe(
    train, columna_texto="text", n_ngramas=N_NGRAMAS, metodo_raiz=METODO_RAIZ
)
test_limpio = procesar_dataframe(
    test, columna_texto="text", n_ngramas=N_NGRAMAS, metodo_raiz=METODO_RAIZ
)

print(f"train_limpio: {train_limpio.shape}")
print(f"test_limpio:  {test_limpio.shape}")


train_limpio: (7613, 18)
test_limpio:  (3263, 17)
CPU times: total: 10.2 s
Wall time: 10.3 s


In [15]:
train_limpio.columns.tolist()


['id',
 'keyword',
 'location',
 'text',
 'target',
 'texto_normalizado',
 'texto_sin_ruido',
 'texto_con_flags',
 'tiene_link',
 'tiene_mencion',
 'tiene_hashtag',
 'menciona_911',
 'tokens',
 'tokens_sin_stopwords',
 'tokens_procesados',
 'ngramas',
 'texto_limpio',
 'texto_ngramas']

### Control de calidad

Se revisa que el pipeline no haya dejado filas vacías, ya que un tweet formado
solo por un link y stopwords puede quedar sin ningún token.


In [16]:
vacios_train = (train_limpio["tokens_procesados"].str.len() == 0).sum()
vacios_test = (test_limpio["tokens_procesados"].str.len() == 0).sum()

print(f"Tweets que quedaron sin tokens en train: {vacios_train}")
print(f"Tweets que quedaron sin tokens en test:  {vacios_test}")

if vacios_train:
    display(train_limpio.loc[train_limpio["tokens_procesados"].str.len() == 0,
                             ["text", "texto_limpio"]].head())


Tweets que quedaron sin tokens en train: 0
Tweets que quedaron sin tokens en test:  1


In [17]:
# Efecto de la limpieza sobre el tamaño del vocabulario
vocab_crudo = set()
for t in train["text"]:
    vocab_crudo.update(str(t).lower().split())

vocab_limpio = set()
for tokens in train_limpio["tokens_procesados"]:
    vocab_limpio.update(tokens)

vocab_ngramas = set()
for ng in train_limpio["ngramas"]:
    vocab_ngramas.update(ng)

resumen_vocab = pd.DataFrame({
    "etapa": ["Texto crudo (split simple)", f"Tokens procesados ({METODO_RAIZ})",
              f"Términos con n-gramas (n<={N_NGRAMAS})"],
    "términos únicos": [len(vocab_crudo), len(vocab_limpio), len(vocab_ngramas)],
})
resumen_vocab


,etapa,términos únicos
0,Texto crudo (split simple),27983
1,Tokens procesados (lema),14579
2,Términos con n-gramas (n<=2),63560


In [18]:
# ¿Cuánta señal aportan los flags nuevos? Se compara su tasa por clase.
flags = pd.DataFrame({
    "tiene_link": train_limpio["tiene_link"],
    "tiene_mencion": train_limpio["tiene_mencion"],
    "tiene_hashtag": train_limpio["tiene_hashtag"],
    "menciona_911": train_limpio["menciona_911"],
    "tiene_algún_número": train_limpio["tokens_procesados"].apply(
        lambda ts: FLAG_NUMERO in ts
    ),
    "target": train_limpio["target"],
})

resumen_flags = (flags.groupby("target").mean() * 100).round(2)
resumen_flags.index = ["No desastre (0)", "Desastre real (1)"]
resumen_flags


,tiene_link,tiene_mencion,tiene_hashtag,menciona_911,tiene_algún_número
No desastre (0),41.43,30.95,20.38,0.02,15.43
Desastre real (1),66.40,20.36,26.26,0.09,24.24


In [19]:
# ¿En cuántos tweets se activa realmente cada flag?
print(f"Tweets que mencionan el 911: {train_limpio['menciona_911'].sum()} de {len(train_limpio)}")
print(f"Tweets con algún número:     {flags['tiene_algún_número'].sum()} de {len(train_limpio)}")
print()
for _, fila in train_limpio[train_limpio["menciona_911"]].iterrows():
    print(f"  [target={fila['target']}] {fila['text'][:100]}")


Tweets que mencionan el 911: 4 de 7613
Tweets con algún número:     1463 de 7613

  [target=1] @_jeesss_ @Ethereal_7 Hello 911 yeah we have someone drowning here send a medic http://t.co/7GiglwdM
  [target=1] Came across this fire video not mine..enjoy..#fire #firemen #firetruck #emergency #rescue #911 #summ
  [target=0] County 911 Overload Prompts Use of Emergency Plan During July 4 Celebrations http://t.co/HXTUPrA5bc 
  [target=1] Kirsten Gillibrand http://t.co/amEA3LaMDj    Extend Health Care To 911 First RESPONDERS !


**Observación honesta sobre el flag del 911:** el mecanismo funciona, pero
en este dataset **solo 4 tweets de 7 613 mencionan el 911** (0.05%). Aunque 3
de esos 4 son desastres reales, con esa frecuencia el flag es estadísticamente
irrelevante: un modelo no puede aprender nada útil de 4 ejemplos. Se conserva
porque no hace daño y porque documenta el criterio, pero **no debe esperarse
que aporte al desempeño**.

El flag genérico de número sí tiene volumen suficiente para ser útil: aparece
en ~19% de los tweets y discrimina razonablemente (24% en desastres reales vs
15% en el resto), probablemente porque las noticias de desastres citan cifras
de víctimas, fechas y magnitudes.


In [20]:
# ¿Qué hashtags y menciones quedaron capturados como bigramas?
from collections import Counter

contador_marcados = Counter()
for tokens in train_limpio["tokens_procesados"]:
    for bigrama in generar_ngramas(tokens, 2):
        if bigrama.startswith((FLAG_HASHTAG, FLAG_MENCION)):
            contador_marcados[bigrama] += 1

pd.DataFrame(contador_marcados.most_common(15),
             columns=["bigrama", "frecuencia"])


,bigrama,frecuencia
0,flagmencion_youtube,83
1,flaghashtag_news,74
2,flaghashtag_flagnumero,44
3,flaghashtag_flaghashtag,37
4,flaghashtag_hot,30
5,flaghashtag_prebreak,30
6,flaghashtag_best,30
7,flaghashtag_job,26
8,flaghashtag_nowplaying,23
9,flaghashtag_hiroshima,23


**Observación:** los bigramas marcados dejan ver de un vistazo qué
hashtags y cuentas son los más usados en el corpus, información que se habría
perdido al quitar el símbolo sin dejar rastro. Además, como el unigrama se
conserva aparte, `earthquake` sigue acumulando frecuencia venga o no de un
hashtag.

**Dos artefactos conocidos de este criterio**, ambos verificados sobre el
corpus y dejados a propósito:

1. `flaghashtag_flaghashtag` aparece 37 veces: son los tweets que terminan con
   varios hashtags seguidos (`#CAfire #wildfires`). No es un error, y de hecho
   señala el uso intensivo de hashtags.
2. La expresión regular de menciones marca como mención cualquier `@palabra`,
   incluido el `@` que se usa para censurar groserías (`f$&@ing` →
   `flagmencion ing`). Esto pasa en **1 solo tweet de 7 613**. Se podría evitar
   exigiendo un espacio antes del `@`, pero eso rompería las **58** menciones
   legítimas que vienen precedidas de puntuación (`.@NorwayMFA`,
   `'@Alexis_Sanchez`), así que se prefiere el criterio actual.


In [21]:
longitudes = pd.DataFrame({
    "palabras_antes": train["text"].str.split().str.len(),
    "tokens_despues": train_limpio["tokens_procesados"].str.len(),
})
longitudes["reduccion_%"] = (
    (1 - longitudes["tokens_despues"] / longitudes["palabras_antes"]) * 100
).round(1)
longitudes.describe().round(2)


,palabras_antes,tokens_despues,reduccion_%
count,7613.00,7613.00,7613.00
mean,14.90,10.38,28.35
std,5.73,4.47,22.94
min,1.00,1.00,-200.00
25%,11.00,7.00,18.20
50%,15.00,10.00,31.20
75%,19.00,13.00,42.90
max,31.00,37.00,87.50


## 4. Guardado en `data/processed`

El nombre del archivo incluye el método de reducción y **el número de n-gramas**
(`train_limpio_lema_2gramas.csv`), de modo que probar con `N_NGRAMAS = 3` genere
un archivo distinto en vez de sobrescribir este.


In [22]:
ruta_train = guardar_procesado(
    train_limpio, DIR_PROCESSED, "train", n_ngramas=N_NGRAMAS, metodo_raiz=METODO_RAIZ
)
ruta_test = guardar_procesado(
    test_limpio, DIR_PROCESSED, "test", n_ngramas=N_NGRAMAS, metodo_raiz=METODO_RAIZ
)

print(f"Guardado: {ruta_train.name}  ({ruta_train.stat().st_size / 1e6:.2f} MB)")
print(f"Guardado: {ruta_test.name}  ({ruta_test.stat().st_size / 1e6:.2f} MB)")


Guardado: train_limpio_lema_2gramas.csv  (9.49 MB)
Guardado: test_limpio_lema_2gramas.csv  (4.12 MB)


In [23]:
# Verificación: se relee el archivo guardado
verificacion = pd.read_csv(ruta_train)
print(f"Filas releídas: {len(verificacion)} (original: {len(train)})")
verificacion[["id", "text", "texto_limpio", "tiene_link", "target"]].head(3)


Filas releídas: 7613 (original: 7613)


,id,text,texto_limpio,tiene_link,target
0,1,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,deed reason flaghashtag earthquake may allah forgive,False,1
1,4,Forest fire near La Ronge Sask. Canada,forest fire near ronge sask canada,False,1
2,5,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place or...,resident ask shelter place notify officer no evacuation shelter place order expect,False,1


## 5. Comparativa antes / después

Comparación fila por fila del tweet original contra el resultado del pipeline.
Se eligen deliberadamente casos difíciles (links, menciones, hashtags,
negaciones, artefactos de codificación) además de una muestra aleatoria.


In [24]:
def comparar(df, indices, titulo):
    print("=" * 100)
    print(titulo)
    print("=" * 100)
    for idx in indices:
        fila = df.loc[idx]
        print(f"\n[id {fila['id']}]  target = {fila['target']}  |  link = {fila['tiene_link']}"
              f"  |  @ = {fila['tiene_mencion']}  |  # = {fila['tiene_hashtag']}"
              f"  |  911 = {fila['menciona_911']}")
        print(f"  ANTES   : {fila['text']}")
        print(f"  DESPUÉS : {fila['texto_limpio']}")
        print(f"  BIGRAMAS: {generar_ngramas(fila['tokens_procesados'], 2)[:6]}")


In [25]:
# Un caso representativo de cada situación difícil, sin repetir tweets
# (un mismo tweet puede traer link y mención a la vez).
criterios = {
    "con link": train_limpio["tiene_link"],
    "con mención": train_limpio["tiene_mencion"],
    "con hashtag": train_limpio["tiene_hashtag"],
    "con 911": train_limpio["menciona_911"],
    "con otros números": train_limpio["tokens_procesados"].apply(
        lambda ts: FLAG_NUMERO in ts and FLAG_EMERGENCIA not in ts
    ),
    "con negación": train_limpio["tokens_procesados"].apply(
        lambda ts: any(t in NEGACIONES for t in ts)
    ),
    "con artefactos de codificación": train_limpio["text"].str.contains(
        r"\x89|Û|å", na=False, regex=True
    ),
}

casos = {}
usados = set()
for etiqueta, mascara in criterios.items():
    disponibles = [i for i in train_limpio[mascara].index if i not in usados]
    if disponibles:
        casos[etiqueta] = disponibles[0]
        usados.add(disponibles[0])

comparar(train_limpio, list(casos.values()),
         "CASOS DIFÍCILES: " + " · ".join(casos.keys()))


CASOS DIFÍCILES: con link · con mención · con hashtag · con 911 · con otros números · con negación · con artefactos de codificación

[id 48]  target = 1  |  link = True  |  @ = True  |  # = False  |  911 = False
  ANTES   : @bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C
  DESPUÉS : flagmencion bbcmtd wholesale market ablaze
  BIGRAMAS: ['flagmencion_bbcmtd', 'bbcmtd_wholesale', 'wholesale_market', 'market_ablaze']

[id 54]  target = 0  |  link = False  |  @ = True  |  # = True  |  911 = False
  ANTES   : @PhDSquares #mufc they've built so much hype around new acquisitions but I doubt they will set the EPL ablaze this season.
  DESPUÉS : flagmencion phdsquares flaghashtag mufc build much hype around new acquisition doubt set epl ablaze season
  BIGRAMAS: ['flagmencion_phdsquares', 'phdsquares_flaghashtag', 'flaghashtag_mufc', 'mufc_build', 'build_much', 'much_hype']

[id 1]  target = 1  |  link = False  |  @ = False  |  # = True  |  911 = False
  ANTES   : Our Deeds are the Rea

In [26]:
# Muestra aleatoria reproducible
muestra = train_limpio.sample(6, random_state=42).index
comparar(train_limpio, muestra, "MUESTRA ALEATORIA (random_state=42)")


MUESTRA ALEATORIA (random_state=42)

[id 3796]  target = 1  |  link = False  |  @ = False  |  # = False  |  911 = False
  ANTES   : So you have a new weapon that can cause un-imaginable destruction.
  DESPUÉS : new weapon cause imaginable destruction
  BIGRAMAS: ['new_weapon', 'weapon_cause', 'cause_imaginable', 'imaginable_destruction']

[id 3185]  target = 0  |  link = False  |  @ = True  |  # = True  |  911 = False
  ANTES   : The f$&amp;@ing things I do for #GISHWHES Just got soaked in a deluge going for pads and tampons. Thx @mishacollins @/@
  DESPUÉS : flagmencion ing thing flaghashtag gishwhes get soaked deluge go pad tampon thx flagmencion mishacollins
  BIGRAMAS: ['flagmencion_ing', 'ing_thing', 'thing_flaghashtag', 'flaghashtag_gishwhes', 'gishwhes_get', 'get_soaked']

[id 7769]  target = 1  |  link = True  |  @ = True  |  # = False  |  911 = False
  ANTES   : DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe CoL police can catch a pickpocket in Liverpool Stree... http://t.co/v

In [27]:
# Misma comparación en formato tabla
tabla_comparativa = train_limpio.loc[muestra, [
    "id", "target", "text", "texto_limpio",
    "tiene_link", "tiene_mencion", "tiene_hashtag",
]].rename(columns={
    "text": "ANTES (crudo)",
    "texto_limpio": "DESPUÉS (limpio)",
})
tabla_comparativa


,id,target,ANTES (crudo),DESPUÉS (limpio),tiene_link,tiene_mencion,tiene_hashtag
2644,3796,1,So you have a new weapon that can cause un-imaginable destruction.,new weapon cause imaginable destruction,False,False,False
2227,3185,0,The f$&amp;@ing things I do for #GISHWHES Just got soaked in a deluge going for pads and tampons. Thx @mishacollins @/@,flagmencion ing thing flaghashtag gishwhes get soaked deluge go pad tampon thx flagmencion mishacollins,False,True,True
5448,7769,1,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe CoL police can catch a pickpocket in Liverpool Stree... http://t.co/v...,flagmencion georgegalloway flagmencion galloway 4mayor uithe col police catch pickpocket liverpool stree,True,True,False
132,191,0,Aftershock back to school kick off was great. I want to thank everyone for making it possible. What a great night.,aftershock back school kick great want thank everyone make possible great night,False,False,False
6845,9810,0,in response to trauma Children of Addicts develop a defensive self - one that decreases vulnerability. (3,response trauma child addict develop defensive self one decrease vulnerability flagnumero flagnoemergencia,False,False,False
5559,7934,0,@Calum5SOS you look like you got caught in a rainstorm this is amazing and disgusting at the same time,flagmencion calum 5sos look like get catch rainstorm amaze disgusting time,False,True,False


In [28]:
# Cuánto se redujo cada uno de esos tweets
detalle = pd.DataFrame({
    "id": train_limpio.loc[muestra, "id"],
    "palabras_antes": train_limpio.loc[muestra, "text"].str.split().str.len(),
    "tokens_despues": train_limpio.loc[muestra, "tokens_procesados"].str.len(),
    "terminos_con_bigramas": train_limpio.loc[muestra, "ngramas"].str.len(),
})
detalle["reduccion_%"] = (
    (1 - detalle["tokens_despues"] / detalle["palabras_antes"]) * 100
).round(1)
detalle


,id,palabras_antes,tokens_despues,terminos_con_bigramas,reduccion_%
2644,3796,11,5,9,54.5
2227,3185,21,14,27,33.3
5448,7769,15,12,23,20.0
132,191,21,12,23,42.9
6845,9810,17,12,23,29.4
5559,7934,19,11,21,42.1


## 6. Resumen

- Se aplicó el pipeline de 7 pasos (`normalización → ruido → números/911 →
  tokenización → stopwords → lematización → n-gramas`) a `train.csv` y
  `test.csv`.
- Los datos procesados quedaron en `data/processed/` con el número de n-gramas
  en el nombre, de modo que probar otro valor de `N_NGRAMAS` no sobrescriba
  este resultado.
- El texto conserva la información que el EDA señaló como útil: palabras de
  hashtags y menciones, el flag de link, las negaciones y los números
  resumidos como flags.
- El flag de número aporta señal (24% vs 15% entre clases), pero el del **911
  se activa en solo 4 tweets**, así que no se espera que influya en el modelo.
- Cada etapa intermedia quedó guardada como columna, lo que permite auditar el
  resultado fila por fila y rehacer solo el tramo final si se cambia de criterio.

**Siguiente paso:** vectorizar `texto_ngramas` con TF-IDF y entrenar el
clasificador, usando F1-score como métrica principal por el desbalance leve de
clases.

> Nota: `data/processed/` está en `.gitignore` (los datos derivados no se
> versionan porque son reproducibles corriendo este notebook). Los datos crudos
> en `data/raw/` sí se conservan en el repositorio.
